In [5]:
### SETUP E TEST

import json
import sys
from pathlib import Path
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_core.documents import Document
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from src.config import (
    EMBEDDING_MODEL,
    QDRANT_URL,
)

COLLECTION_NAME = "ds_armstrong_debug"

# ==========================================
# efinizione dei 10 Chunk Sintetici
# ==========================================
toy_chunks = [
    # --- JAZZ (Louis Armstrong) ---[cite: 1]
    Document(
        page_content="Louis Armstrong, soprannominato Satchmo, è stato uno dei più celebri musicisti e trombettisti jazz della storia.",
        metadata={
            "chunk_id": "armstrong_jazz_01",
            "topic": "jazz",
            "person": "Louis Armstrong",
        },
    ),
    Document(
        page_content="Nato a New Orleans, Louis Armstrong ha rivoluzionato il genere del jazz con la sua voce rauca e le sue innovazioni alla tromba.",
        metadata={
            "chunk_id": "armstrong_jazz_02",
            "topic": "jazz",
            "person": "Louis Armstrong",
        },
    ),
    Document(
        page_content="Tra le canzoni più celebri di Louis Armstrong ricordiamo 'What a Wonderful World' e 'Hello, Dolly!', capolavori della musica jazz.",
        metadata={
            "chunk_id": "armstrong_jazz_03",
            "topic": "jazz",
            "person": "Louis Armstrong",
        },
    ),
    Document(
        page_content="La storia del jazz americano trova in Louis Armstrong una delle sue figure più iconiche ed influenti del Novecento.",
        metadata={
            "chunk_id": "armstrong_jazz_04",
            "topic": "jazz",
            "person": "Louis Armstrong",
        },
    ),
    # --- CICLISMO (Lance Armstrong) ---[cite: 1]
    Document(
        page_content="Lance Armstrong è stato un famoso ciclista su strada statunitense, noto per aver vinto sette edizioni consecutive del Tour de France.",
        metadata={
            "chunk_id": "armstrong_bike_01",
            "topic": "ciclismo",
            "person": "Lance Armstrong",
        },
    ),
    Document(
        page_content="Il mondo del ciclismo professionistico è stato profondamente segnato dalle vicende agonistiche e sportive legate a Lance Armstrong.",
        metadata={
            "chunk_id": "armstrong_bike_02",
            "topic": "ciclismo",
            "person": "Lance Armstrong",
        },
    ),
    Document(
        page_content="Nelle corse a tappe in bicicletta, il ciclista texano Armstrong si distinse per le sue doti eccezionali in salita e a cronometro.",
        metadata={
            "chunk_id": "armstrong_bike_03",
            "topic": "ciclismo",
            "person": "Lance Armstrong",
        },
    ),
    # --- ASTRONAUTA / SPAZIO (Neil Armstrong) ---[cite: 1]
    Document(
        page_content="Neil Armstrong è stato un astronauta e aviatore statunitense, il primo uomo a camminare sulla Luna durante la missione Apollo 11 nel 1969.",
        metadata={
            "chunk_id": "armstrong_space_01",
            "topic": "spazio",
            "person": "Neil Armstrong",
        },
    ),
    Document(
        page_content="Con la celebre frase 'Un piccolo passo per un uomo, un grande balzo per l'umanità', l'astronauta Neil Armstrong sbarcò sul suolo lunare.",
        metadata={
            "chunk_id": "armstrong_space_02",
            "topic": "spazio",
            "person": "Neil Armstrong",
        },
    ),
    Document(
        page_content="La NASA selezionò Neil Armstrong come comandante del modulo lunare Eagle per la prima e storica esplorazione della superficie della Luna.",
        metadata={
            "chunk_id": "armstrong_space_03",
            "topic": "spazio",
            "person": "Neil Armstrong",
        },
    ),
]


def setup_and_test():
    # ==========================================
    # Inizializzazione Client ed Embedding
    # ==========================================
    client = QdrantClient(url=QDRANT_URL)
    embeddings = FastEmbedEmbeddings(model_name=EMBEDDING_MODEL)

    # Calcolo dimensione dinamica del vettore
    vector_dim = len(embeddings.embed_query("test"))

    # Pulizia e ricreazione collezione[cite: 3, 4]
    if client.collection_exists(COLLECTION_NAME):
        client.delete_collection(COLLECTION_NAME)

    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=vector_dim, distance=Distance.COSINE),
    )

    vectorstore = QdrantVectorStore(
        client=client,
        collection_name=COLLECTION_NAME,
        embedding=embeddings,
    )

    # Caricamento dei 10 chunk[cite: 3, 4]
    vectorstore.add_documents(documents=toy_chunks)
    print(
        f"!>> Inseriti {len(toy_chunks)} chunk nella collezione di test '{COLLECTION_NAME}'!\n"
    )

    # ==========================================
    # Test di Retrieval Semantico (Query Ambigua)
    # ==========================================
    query_ambigua = "Chi era Armstrong e quali sono i suoi successi?"
    print(f"?> Esecuzione Query: '{query_ambigua}'")
    print("=" * 80)

    results = vectorstore.similarity_search_with_score(
        query=query_ambigua, k=6
    )

    for idx, (doc, score) in enumerate(results, 1):
        topic = doc.metadata.get("topic")
        person = doc.metadata.get("person")
        chunk_id = doc.metadata.get("chunk_id")
        print(f"[{idx}] Score Coseno: {score:.4f} | Tema: {topic.upper()} ({person}) | ID: {chunk_id}")
        print(f"    Snippet: {doc.page_content}\n")


if __name__ == "__main__":
    setup_and_test()

!>> Inseriti 10 chunk nella collezione di test 'ds_armstrong_debug'!

?> Esecuzione Query: 'Chi era Armstrong e quali sono i suoi successi?'
[1] Score Coseno: 0.7419 | Tema: SPAZIO (Neil Armstrong) | ID: armstrong_space_01
    Snippet: Neil Armstrong è stato un astronauta e aviatore statunitense, il primo uomo a camminare sulla Luna durante la missione Apollo 11 nel 1969.

[2] Score Coseno: 0.7295 | Tema: JAZZ (Louis Armstrong) | ID: armstrong_jazz_01
    Snippet: Louis Armstrong, soprannominato Satchmo, è stato uno dei più celebri musicisti e trombettisti jazz della storia.

[3] Score Coseno: 0.7206 | Tema: SPAZIO (Neil Armstrong) | ID: armstrong_space_02
    Snippet: Con la celebre frase 'Un piccolo passo per un uomo, un grande balzo per l'umanità', l'astronauta Neil Armstrong sbarcò sul suolo lunare.

[4] Score Coseno: 0.7149 | Tema: CICLISMO (Lance Armstrong) | ID: armstrong_bike_02
    Snippet: Il mondo del ciclismo professionistico è stato profondamente segnato dalle vicende agon

In [3]:
### TEST PIPELINE -- istanziazione classe
import sys
from pathlib import Path
import networkx as nx
import numpy as np
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_core.documents import Document
from langchain_ollama import ChatOllama
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.chunk_widget import ChunkGraphWidget
from src.config import (
    BASE_DISTANCE_PX,
    EMBEDDING_MODEL,
    LLM_MODEL,
    QDRANT_URL,
    RETRIEVAL_TOP_K,
)
from src.distance_reranker import DistanceReranker
from src.graph_builder import KnowledgeGraphBuilder
from src.sidecar_manager import SidecarManager


class ArmstrongPipelineTester:

    def __init__(
        self,
        qdrant_url: str = QDRANT_URL,
        collection_name: str = "ds_armstrong_debug",
        embedding_model: str = EMBEDDING_MODEL,
        llm_model: str = LLM_MODEL,
        base_distance_px: float = BASE_DISTANCE_PX,
        sidecar_path: Path
        | str = project_root / "data/processed/test/sidecar_04_02T.json",
    ):
        """Inizializza la pipeline connettendosi alla collezione Qdrant e ai moduli di re-ranking."""
        self.collection_name = collection_name
        self.client = QdrantClient(url=qdrant_url)
        self.embeddings = FastEmbedEmbeddings(model_name=embedding_model)
        self.llm = ChatOllama(model=llm_model, temperature=0.0)
        self.base_distance_px = base_distance_px

        # Connessione al Vector Store
        self.vector_store = QdrantVectorStore(
            client=self.client,
            collection_name=self.collection_name,
            embedding=self.embeddings,
        )

        # Inizializzazione dei moduli Sidecar e DistanceReranker
        self.sidecar_manager = SidecarManager(filepath=Path(sidecar_path))
        self.reranker = DistanceReranker(sidecar_manager=self.sidecar_manager)

    def retrieve_and_build_graph(
        self, query: str, top_k: int = RETRIEVAL_TOP_K, tau: float = 0.50
    ):
        """Retrieval vettoriale e popolamento del grafo tramite KnowledgeGraphBuilder."""
        retrieved_results = self.vector_store.similarity_search_with_score(
            query=query, k=top_k
        )

        builder = KnowledgeGraphBuilder()
        doc_vectors = []
        doc_list = []

        for doc, score in retrieved_results:
            c_id = doc.metadata.get("chunk_id", "unknown_id")
            topic = doc.metadata.get("topic", "general")
            person = doc.metadata.get("person", "unknown")

            builder.add_chunk_node(
                chunk_id=c_id, text=doc.page_content
            )

            vec = self.embeddings.embed_query(doc.page_content)
            doc_vectors.append(vec)
            doc_list.append(c_id)

        # Calcolo delle relazioni di similarità coseno tra i chunk
        num_docs = len(doc_list)
        for i in range(num_docs):
            for j in range(i + 1, num_docs):
                id_i, id_j = doc_list[i], doc_list[j]
                v_i, v_j = np.array(doc_vectors[i]), np.array(doc_vectors[j])

                cos_sim = float(
                    np.dot(v_i, v_j)
                    / (np.linalg.norm(v_i) * np.linalg.norm(v_j))
                )

                if cos_sim >= tau:
                    builder.add_relation(
                        source_id=id_i, target_id=id_j, weight=cos_sim
                    )

        return builder, retrieved_results

    def apply_reranking(self, retrieved_results, top_n: int = 4):
        """Fase 2: Delega il re-ranking alla classe DistanceReranker."""
        return self.reranker.rerank(candidates=retrieved_results, top_n=top_n)

    def generate_response(self, query: str, candidates: list):
        """Fase 3: Formatta il contesto riordinato ed esegue la query su Ollama."""
        formatted_context_list = []

        for item in candidates:
            # Gestione sicura sia dell'output di DistanceReranker (dict) che dei tuple (doc, score)
            if isinstance(item, dict) and "raw_doc" in item:
                doc = item["raw_doc"]
            elif isinstance(item, tuple):
                doc = item[0]
            else:
                doc = item

            c_id = doc.metadata.get("chunk_id", "chunk")
            person = doc.metadata.get("person", "")
            formatted_context_list.append(
                f"[{c_id}] ({person}): {doc.page_content}"
            )

        formatted_context = "\n\n".join(formatted_context_list)

        prompt = f"""Sei un assistente preciso. Rispondi alla domanda usando ESCLUSIVAMENTE le informazioni presenti nel contesto fornito.
Se il contesto è vuoto o non contiene informazioni sufficienti, dichiara chiaramente che non puoi rispondere.

--- CONTESTO ---
{formatted_context}

--- DOMANDA ---
{query}

--- RISPOSTA ---"""

        response = self.llm.invoke(prompt)
        return response.content

In [4]:
# 1. Istanziazione della pipeline
tester = ArmstrongPipelineTester(collection_name="ds_armstrong_debug")

query = "Quali sono stati i principali successi ottenuti da Armstrong?"

# 2. Retrieval e Grafo
builder, initial_results = tester.retrieve_and_build_graph(
    query=query, top_k=6, tau=0.50
)

# Render del grafo
widget = ChunkGraphWidget()
widget.graph_data = builder.to_json_data()
display(widget)

# 3. Test Re-ranking baseline (senza variazioni salvate nel sidecar)
reranked_baseline = tester.apply_reranking(initial_results, top_n=4)
print("--- RISPOSTA BASELINE (R_t) ---")
print(tester.generate_response(query, reranked_baseline))

# 4. Simulazione salvataggio delta nel Sidecar (es. l'utente allontana i chunk di ciclismo/spazio)
# In un'esecuzione reale, questo JSON viene aggiornato dall'interfaccia grafica.
sidecar_data = {
    "pairwise_deltas": {
        "armstrong_bike_01_AND_armstrong_jazz_01": {
            "distance_factor": 3.0
        },  # Supera hard_filter_threshold
        "armstrong_space_01_AND_armstrong_jazz_01": {
            "distance_factor": 3.0
        },
    }
}
tester.sidecar_manager.save_data(sidecar_data)

# 5. Esecuzione Re-ranking reattivo con i delta del Sidecar
reranked_adapted = tester.apply_reranking(initial_results, top_n=4)
print("\n--- RISPOSTA ADATTATA (R_{t+1}) ---")
print(tester.generate_response(query, reranked_adapted))

TypeError: KnowledgeGraphBuilder.add_relation() got an unexpected keyword argument 'weight'